In [ ]:
#Setup
#if tou run it through colab
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scvi
from pathlib import Path

scvi.settings.seed = 0
np.random.seed(0)
sc.settings.verbosity = 1

DATA = Path("/content/drive/MyDrive/POI/data")
CKPT = Path("/content/drive/MyDrive/POI/checkpoints")
MODELS = Path("/content/drive/MyDrive/POI/models")
CKPT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

In [ ]:
#Setup
#if you run it local
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scvi

from pathlib import Path

# reproducibility
scvi.settings.seed = 0
np.random.seed(0)

sc.settings.verbosity = 1
DATA = Path("/home/frag/Desktop/POI/data")
CKPT = Path("/home/frag/Desktop/POI/checkpoints")
MODELS = Path("/home/frag/Desktop/POI/models")

CKPT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

In [ ]:
#files case & control
case_files = {
    "G1": DATA / "GSM8281749_G1_filtered_feature_bc_matrix.h5",
    "G2": DATA / "GSM8281750_G2_filtered_feature_bc_matrix.h5",
    "G3": DATA / "GSM8281751_G3_filtered_feature_bc_matrix.h5",
    "G4": DATA / "GSM8281752_G4_filtered_feature_bc_matrix.h5",
    "G5": DATA / "GSM8281753_G5_filtered_feature_bc_matrix.h5",
}

control_files = {
    "C3": DATA / "GSM8281754_C3_filtered_feature_bc_matrix.h5",
    "C4": DATA / "GSM8281755_C4_filtered_feature_bc_matrix.h5",
    "C5": DATA / "GSM8281756_C5_filtered_feature_bc_matrix.h5",
    "C6": DATA / "GSM8281757_C6_filtered_feature_bc_matrix.h5",
}
all_files = {**{k: (v, "case") for k, v in case_files.items()},
             **{k: (v, "control") for k, v in control_files.items()}}

In [ ]:
#SOLO doublet detection = HVG selection -> scVI training -> solo classifier -> score
def run_solo_for_sample(path, sample_name):

    ckpt_file = CKPT / f"solo_scores_{sample_name}.csv"

    if ckpt_file.exists():
        print(f"[{sample_name}] Found checkpoint, loading scores.")
        return pd.read_csv(ckpt_file, index_col=0)

    print(f"[{sample_name}] No checkpoint found, running scVI + SOLO...")

    a = sc.read_10x_h5(path)
    a.var_names_make_unique()

    a_hvg = a.copy()
    sc.pp.filter_genes(a_hvg, min_cells=10)
    sc.pp.highly_variable_genes(
        a_hvg, n_top_genes=2000, subset=True, flavor="seurat_v3"
    )

    scvi.model.SCVI.setup_anndata(a_hvg)
    vae = scvi.model.SCVI(a_hvg)
    vae.train()

    solo = scvi.external.SOLO.from_scvi_model(vae)
    solo.train()

    df = solo.predict()
    df["prediction"] = solo.predict(soft=False)

    df.index = df.index.str.replace(r"-\d+$", "", regex=True)
    df["dif"] = df["doublet"] - df["singlet"]

    df.to_csv(ckpt_file)
    print(f"[{sample_name}] Saved checkpoint: {ckpt_file}")

    return df

In [ ]:
#Run SOLO
solo_scores = {}

for sample_name, (path, group) in all_files.items():
    solo_scores[sample_name] = run_solo_for_sample(path, sample_name)

In [ ]:
#Plot singlet/doublet split
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for ax, (sample_name, df) in zip(axes, solo_scores.items()):
    ax.hist(df.loc[df["prediction"] == "singlet", "dif"],
            bins=50, alpha=0.6, label="singlet", color="steelblue")
    ax.hist(df.loc[df["prediction"] == "doublet", "dif"],
            bins=50, alpha=0.6, label="doublet", color="orangered")
    ax.axvline(0, color="white", linestyle="--", linewidth=0.8)
    ax.set_title(sample_name)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
#Plot subset "doublet"
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for ax, (sample_name, df) in zip(axes, solo_scores.items()):
    doublet_difs = df.loc[df["prediction"] == "doublet", "dif"]
    ax.hist(doublet_difs, bins=50)
    ax.set_title(sample_name)
    ax.set_xlabel("dif")

plt.tight_layout()
plt.show()

In [ ]:
#Doublet removal per sample
def load_sample_clean(file, sample, group, solo_df, threshold=0.5):
    adata_s = sc.read_10x_h5(file)
    adata_s.var_names_make_unique()

    clean_barcodes = adata_s.obs_names.str.replace(r"-\d+$", "", regex=True)

    doublet_barcodes = solo_df.loc[
        (solo_df["prediction"] == "doublet") & (solo_df["dif"] > threshold)
    ].index

    n_before = adata_s.n_obs
    keep_mask = ~clean_barcodes.isin(doublet_barcodes)
    adata_s = adata_s[keep_mask].copy()
    n_after = adata_s.n_obs
    print(f"[{sample}] removed {n_before - n_after} doublets "
          f"({(n_before-n_after)/n_before*100:.1f}%), kept {n_after}")

    adata_s.obs_names = [f"{sample}_{x}" for x in adata_s.obs_names]
    adata_s.obs["sample"] = sample
    adata_s.obs["group"] = group
    return adata_s

In [ ]:
#Concat
datasets = []
for sample, file in case_files.items():
    datasets.append(load_sample_clean(file, sample, "case", solo_scores[sample]))
for sample, file in control_files.items():
    datasets.append(load_sample_clean(file, sample, "control", solo_scores[sample]))

adata = sc.concat(datasets, join="outer")
adata.var_names_make_unique()
print("Shape:", adata.shape)
print("Duplicated var_names:", adata.var_names.duplicated().sum())

In [ ]:
#QC metrics
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = (
    adata.var_names.str.startswith("RPS") |
    adata.var_names.str.startswith("RPL")
)
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], percent_top=None, inplace=True)

adata.obs.head()

In [ ]:
#Violin plot for visual checking
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
    jitter=0.4,
    multi_panel=True
)

In [ ]:
#QC bar plot per sample
qc = (
    adata.obs
    .groupby("sample")[
        ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"]
    ]
    .mean()
)

qc.plot(
    kind="bar",
    subplots=True,
    figsize=(10, 8),
    layout=(2, 2),
    sharex=True
)

plt.tight_layout()
plt.show()

In [ ]:
#QC filtering
adata_before_qc = adata.copy()

adata = adata_before_qc[
    (adata_before_qc.obs["n_genes_by_counts"] >= 300) &
    (adata_before_qc.obs["n_genes_by_counts"] <= 6000) &
    (adata_before_qc.obs["pct_counts_mt"] < 8) &
    (adata_before_qc.obs["pct_counts_ribo"] < 10)
].copy()

print("Shape before:", adata_before_qc.shape)
print("Shape after:", adata.shape)

In [ ]:
#scVI integration setup = Raw counts per layer, batch-aware HVG + setup.
adata_scvi = adata.copy()

sc.pp.filter_genes(adata_scvi, min_cells=20)

sc.pp.highly_variable_genes(
    adata_scvi,
    n_top_genes=3000,
    subset=True,
    flavor="seurat_v3",
    batch_key="sample"
)

adata_scvi.layers["counts"] = adata_scvi.X.copy()

scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer="counts",
    batch_key="sample"
)

vae = scvi.model.SCVI(adata_scvi, n_latent=30)
vae.train()

In [ ]:
#Clustering & UMAP
adata_scvi.obsm["X_scVI"] = vae.get_latent_representation()

sc.pp.neighbors(adata_scvi, use_rep="X_scVI")
sc.tl.umap(adata_scvi)
sc.tl.leiden(adata_scvi, resolution=0.4, random_state=0)

sc.pl.umap(adata_scvi, color=["leiden", "group", "sample"])

In [ ]:
#Cluster counts = sample-dominated clusters
adata_scvi.obs["leiden"].value_counts()
pd.crosstab(adata_scvi.obs["leiden"], adata_scvi.obs["group"])
pd.crosstab(adata_scvi.obs["leiden"], adata_scvi.obs["sample"])

In [ ]:
#Marker genes
adata_marker = adata_scvi.copy()

sc.pp.normalize_total(adata_marker, target_sum=1e4)
sc.pp.log1p(adata_marker)

sc.tl.rank_genes_groups(
    adata_marker,
    groupby="leiden",
    method="wilcoxon"
)

In [ ]:
#Marker inspection & annotation
#for c in ["3", "4", "7"]:
#    print(f"\nCluster {c}")
#    display(sc.get.rank_genes_groups_df(adata_marker, group=c).head(20))

sc.pl.violin(
    adata_scvi,
    ["n_genes_by_counts",
     "total_counts",
     "pct_counts_mt"],
    groupby="leiden"
)
markers = sc.get.rank_genes_groups_df(
    adata_marker,
    group=None
)
markers.head(50)

In [ ]:
#Markers per cluster (top 10)
for c in adata_marker.obs["leiden"].cat.categories:
    print("\nCLUSTER", c)
    print(
        markers[markers["group"] == c]
        .head(10)[["names", "scores"]]
    )

In [ ]:
#Marker genes per Leiden cluster (Wilcoxon)
sc.pl.rank_genes_groups(
    adata_marker,
    n_genes=20,
    sharey=False
)

In [ ]:
#Annotation

cluster_to_celltype = {
    "0":  "Stromal/Fibroblast",
    "1":  "Endothelial",
    "2":  "PAX8+ epithelial-like / Unresolved",
    "3":  "Stromal/Fibroblast",
    "4":  "Stromal/Theca-like",
    "5":  "Theca/Stromal",
    "6":  "Vascular smooth muscle / Myoid",
    "7":  "Endothelial",
    "8":  "Stromal/Theca-like",
    "9":  "Endothelial",
    "10": "Basement membrane/Stromal",
    "11": "Smooth muscle / Myoid",
    "12": "Immune",
    "13": "Neural",
}

adata_marker.obs["cell_type"] = (
    adata_marker.obs["leiden"]
    .map(cluster_to_celltype)
    .astype("category")
)

adata_scvi.obs["cell_type"] = adata_marker.obs["cell_type"].values
adata_scvi.obs["is_unresolved_cluster"] = adata_scvi.obs["leiden"] == "2"


print(adata_scvi.obs.groupby("cell_type")["leiden"].first())
print(adata_scvi.obs["cell_type"].value_counts())

In [ ]:
pip install adjustText --break-system-packages

In [ ]:
#UMAP:labeled clusters
import matplotlib.pyplot as plt

adata_scvi.obs["cell_type_labeled"] = (
    adata_scvi.obs["leiden"].astype(str) + " - " + adata_scvi.obs["cell_type"].astype(str)
)

ordered_categories = [
    f"{i} - {cluster_to_celltype[str(i)]}"
    for i in range(14)
]

adata_scvi.obs["cell_type_labeled"] = pd.Categorical(
    adata_scvi.obs["cell_type_labeled"],
    categories=ordered_categories,
    ordered=True
)

umap_coords = adata_scvi.obsm["X_umap"]

fig, ax = plt.subplots(figsize=(11, 8))

sc.pl.umap(
    adata_scvi,
    color=["cell_type_labeled"],
    legend_loc="right margin",
    legend_fontsize=7,
    frameon=False,
    ax=ax,
    show=False,
)

manual_offsets = {
    "13": (0.5, 0.2),
    "11": (0.4, 0.3),
}

for cluster_id in adata_scvi.obs["leiden"].cat.categories:
    mask = adata_scvi.obs["leiden"] == cluster_id
    centroid_x = umap_coords[mask, 0].mean()
    centroid_y = umap_coords[mask, 1].mean()

    if cluster_id in manual_offsets:
        dx, dy = manual_offsets[cluster_id]
        label_x, label_y = centroid_x + dx, centroid_y + dy

        ax.annotate(
            cluster_id,
            xy=(centroid_x, centroid_y),
            xytext=(label_x, label_y),
            fontsize=11, fontweight="bold",
            ha="center", va="center",
              bbox=dict(boxstyle="circle,pad=0.15", facecolor="white", alpha=0.4, linewidth=0.5),
            color="black",
        )
    else:
        ax.text(
            centroid_x, centroid_y, cluster_id,
            fontsize=11, fontweight="bold",
            ha="center", va="center",
              bbox=dict(boxstyle="circle,pad=0.15", facecolor="white", alpha=0.4, linewidth=0.5),
            color="black",
        )

plt.tight_layout()
plt.show()

In [ ]:
#Pseudobulk aggregation (sample × cell_type) for per-cell-type DE
adata_pb = adata_scvi.copy()

pb_counts = {}
pb_meta = []

for (sample, ctype), idx in adata_pb.obs.groupby(["sample", "cell_type"]).groups.items():
    n_cells = len(idx)
    if n_cells < 10:
        continue
    counts_sum = np.asarray(adata_pb[idx].layers["counts"].sum(axis=0)).flatten()
    pb_counts[f"{sample}_{ctype}"] = counts_sum
    pb_meta.append({
        "sample_celltype": f"{sample}_{ctype}",
        "sample": sample,
        "cell_type": ctype,
        "group": adata_pb.obs.loc[idx, "group"].iloc[0],
        "n_cells": n_cells,
    })

pb_matrix = pd.DataFrame(pb_counts, index=adata_pb.var_names).T
pb_meta_df = pd.DataFrame(pb_meta).set_index("sample_celltype")

pb_matrix.to_csv(CKPT / "pseudobulk_counts.csv")
pb_meta_df.to_csv(CKPT / "pseudobulk_metadata.csv")

pb_matrix.shape, pb_meta_df.shape

In [ ]:
#Sanity check group counts
pb_meta_df.groupby(["cell_type", "group"]).size().unstack(fill_value=0)

In [ ]:
#Exclude low-quality/unresolved clusters
excluded_celltypes = [
    "PAX8+ epithelial-like / Unresolved",
]
pb_meta_df_filt = pb_meta_df[~pb_meta_df["cell_type"].isin(excluded_celltypes)]
pb_matrix_filt = pb_matrix.loc[pb_meta_df_filt.index]
pb_meta_df_filt.groupby(["cell_type", "group"]).size().unstack(fill_value=0)

In [ ]:
pip install pydeseq2 --break-system-packages

In [ ]:
#sample size & gene count check per cell type before DE
for ctype in pb_meta_df_filt["cell_type"].unique():
    print("="*60)
    print(ctype)
    meta_sub = pb_meta_df_filt[
        pb_meta_df_filt["cell_type"] == ctype
    ].copy()
    counts_sub = pb_matrix_filt.loc[meta_sub.index]
    print("samples:", counts_sub.shape[0])
    print("genes:", counts_sub.shape[1])
    print(meta_sub["group"].value_counts())
    gene_mask = counts_sub.sum(axis=0) >= 10
    print("genes kept:", gene_mask.sum())

In [ ]:
#Per-cell-type DESeq2 (Rows = samples & Columns = genes)
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

de_results = {}

for ctype in pb_meta_df_filt["cell_type"].unique():
    meta_sub = pb_meta_df_filt[pb_meta_df_filt["cell_type"] == ctype].copy()
    counts_sub = pb_matrix_filt.loc[meta_sub.index]

    # skip αν δεν υπάρχουν αρκετά replicates ανά group
    group_counts = meta_sub["group"].value_counts()
    if (group_counts < 2).any() or group_counts.shape[0] < 2:
        print(f"Skipping {ctype}: not enough replicates in each group")
        continue

    gene_mask = counts_sub.sum(axis=0) >= 10
    counts_sub = counts_sub.loc[:, gene_mask]

    meta_sub["group"] = pd.Categorical(
        meta_sub["group"],
        categories=["control", "case"]
    )

    dds = DeseqDataSet(
        counts=counts_sub.astype(int),
        metadata=meta_sub,
        design_factors="group",
        refit_cooks=True,
    )
    dds.deseq2()

    stat_res = DeseqStats(
        dds,
        contrast=["group", "case", "control"]
    )
    stat_res.summary()
    stat_res.lfc_shrink(
        coeff="group[T.case]",
        adapt=False
    )

    res_df = stat_res.results_df.sort_values("padj")
    de_results[ctype] = res_df
    res_df.to_csv(
        CKPT / f"de_{ctype.replace(' ', '_').replace('/', '_')}.csv"
    )
    print(f"{ctype}: {(res_df['padj'] < 0.05).sum()} significant genes")

In [ ]:
## Overall DE: sample-level pseudobulk
# (rows = samples (G1,G2,G3,G4,G5,C3,C4,C5,C6) & columns = genes)
sample_level_counts = pb_matrix_filt.groupby(pb_meta_df_filt["sample"]).sum()
sample_level_meta = pd.DataFrame({
    "group": pb_meta_df_filt.groupby("sample")["group"].first()
})

gene_mask = sample_level_counts.sum(axis=0) >= 10
counts_sub = sample_level_counts.loc[:, gene_mask]
meta_sub = sample_level_meta.loc[counts_sub.index].copy()
meta_sub["group"] = pd.Categorical(meta_sub["group"], categories=["control", "case"])

dds = DeseqDataSet(counts=counts_sub.astype(int), metadata=meta_sub, design_factors="group", refit_cooks=True)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["group", "case", "control"])
stat_res.summary()
stat_res.lfc_shrink(coeff="group[T.case]", adapt=False)

res_df = stat_res.results_df.sort_values("padj")
res_df.to_csv(CKPT / "de_overall.csv")
print(f"Overall: {(res_df['padj'] < 0.05).sum()} significant genes")

In [ ]:
#Dataset without G4 (sensitivity analysis setup)
adata_de = adata_scvi[adata_scvi.obs["sample"] != "G4"].copy()

controls = ["C3", "C4", "C5", "C6"]
cases = ["G1", "G2", "G3", "G5"]

adata_de.obs["group"] = adata_de.obs["sample"].map(
    lambda x: "control" if x in controls else "case"
)

In [ ]:
#Pseudobulk aggregation (no G4) sample × cell_type pseudobulk
pb_counts_de = {}
pb_meta_de = []

for (sample, ctype), idx in adata_de.obs.groupby(["sample", "cell_type"]).groups.items():
    n_cells = len(idx)
    if n_cells < 10:
        continue
    counts_sum = np.asarray(adata_de[idx].layers["counts"].sum(axis=0)).flatten()
    pb_counts_de[f"{sample}_{ctype}"] = counts_sum
    pb_meta_de.append({
        "sample_celltype": f"{sample}_{ctype}",
        "sample": sample,
        "cell_type": ctype,
        "group": adata_de.obs.loc[idx, "group"].iloc[0],
        "n_cells": n_cells,
    })

pb_matrix_de = pd.DataFrame(pb_counts_de, index=adata_de.var_names).T
pb_meta_de_df = pd.DataFrame(pb_meta_de).set_index("sample_celltype")

pb_matrix_de.to_csv(CKPT / "pseudobulk_counts_noG4.csv")
pb_meta_de_df.to_csv(CKPT / "pseudobulk_metadata_noG4.csv")

pb_matrix_de.shape, pb_meta_de_df.shape

In [ ]:
#Sanity check + exclude celltypes (no G4)
pb_meta_de_df.groupby(["cell_type", "group"]).size().unstack(fill_value=0)

excluded_celltypes = [
    "PAX8+ epithelial-like / Unresolved",
]
pb_meta_de_filt = pb_meta_de_df[~pb_meta_de_df["cell_type"].isin(excluded_celltypes)]
pb_matrix_de_filt = pb_matrix_de.loc[pb_meta_de_filt.index]
pb_meta_de_filt.groupby(["cell_type", "group"]).size().unstack(fill_value=0)

In [ ]:
#Per-cell-type DESeq2 (no G4)
de_results_noG4 = {}

for ctype in pb_meta_de_filt["cell_type"].unique():
    meta_sub = pb_meta_de_filt[pb_meta_de_filt["cell_type"] == ctype].copy()
    counts_sub = pb_matrix_de_filt.loc[meta_sub.index]

    group_counts = meta_sub["group"].value_counts()
    if (group_counts < 2).any() or group_counts.shape[0] < 2:
        print(f"Skipping {ctype}: not enough replicates in each group")
        continue

    gene_mask = counts_sub.sum(axis=0) >= 10
    counts_sub = counts_sub.loc[:, gene_mask]

    meta_sub["group"] = pd.Categorical(
        meta_sub["group"],
        categories=["control", "case"]
    )

    dds = DeseqDataSet(
        counts=counts_sub.astype(int),
        metadata=meta_sub,
        design_factors="group",
        refit_cooks=True,
    )
    dds.deseq2()

    stat_res = DeseqStats(
        dds,
        contrast=["group", "case", "control"]
    )
    stat_res.summary()
    stat_res.lfc_shrink(
        coeff="group[T.case]",
        adapt=False
    )

    res_df = stat_res.results_df.sort_values("padj")
    de_results_noG4[ctype] = res_df
    res_df.to_csv(
        CKPT / f"de_noG4_{ctype.replace(' ', '_').replace('/', '_')}.csv"
    )
    print(f"{ctype}: {(res_df['padj'] < 0.05).sum()} significant genes")

In [ ]:
# Overall DE (noG4): sample-level pseudobulk (rows = samples (G1,G2,G3,G4,G5,C3,C4,C5,C6) & columns = genes)

sample_level_counts_noG4 = pb_matrix_de_filt.groupby(pb_meta_de_filt["sample"]).sum()
sample_level_meta_noG4 = pd.DataFrame({
    "group": pb_meta_de_filt.groupby("sample")["group"].first()
})

gene_mask = sample_level_counts_noG4.sum(axis=0) >= 10
counts_sub = sample_level_counts_noG4.loc[:, gene_mask]
meta_sub = sample_level_meta_noG4.loc[counts_sub.index].copy()
meta_sub["group"] = pd.Categorical(meta_sub["group"], categories=["control", "case"])

dds = DeseqDataSet(
    counts=counts_sub.astype(int),
    metadata=meta_sub,
    design_factors="group",
    refit_cooks=True,
)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["group", "case", "control"])
stat_res.summary()
stat_res.lfc_shrink(coeff="group[T.case]", adapt=False)

res_df = stat_res.results_df.sort_values("padj")
res_df.to_csv(CKPT / "de_overall_noG4.csv")
print(f"Overall (noG4): {(res_df['padj'] < 0.05).sum()} significant genes")

In [ ]:
#Top genes barplot
de_overall = pd.read_csv(CKPT / "de_overall.csv", index_col=0)
top_genes = de_overall[de_overall["padj"] < 0.05].sort_values("padj").head(10).index.tolist()
print(top_genes)

#pseudobulk row per sample)
sample_level_counts = pb_matrix_filt.groupby(pb_meta_df_filt["sample"]).sum()
sample_level_meta = pb_meta_df_filt.groupby("sample")["group"].first()

# CPM normalization
cpm = sample_level_counts.div(sample_level_counts.sum(axis=1), axis=0) * 1e6
cpm_log = np.log1p(cpm)

print(sample_level_counts.shape)

In [ ]:
#Barplot top genes (G4 in red for visual outlier check)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for ax, gene in zip(axes, top_genes):
    if gene not in cpm_log.columns:
        continue
    values = cpm_log[gene]
    colors = ["red" if s == "G4" else ("orange" if sample_level_meta[s] == "case" else "steelblue")
              for s in values.index]
    ax.bar(values.index, values.values, color=colors)
    ax.set_title(gene, fontsize=10)
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
#PCA setup sample-level pseudobulk (CPM, standardized)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Χρησιμοποιούμε το ίδιο sample_level_counts / cpm_log από πριν
# Κράτα μόνο genes με reasonable variance (π.χ. filter πρώτα με το ίδιο gene_mask λογικό)
gene_mask = sample_level_counts.sum(axis=0) >= 10
cpm_log_filt = cpm_log.loc[:, gene_mask]

# Standardize πριν το PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cpm_log_filt)

pca = PCA(n_components=5)
pcs = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    pcs[:, :2],
    columns=["PC1", "PC2"],
    index=cpm_log_filt.index
)
pca_df["group"] = sample_level_meta.loc[pca_df.index]
pca_df["sample"] = pca_df.index

print("Explained variance ratio:", pca.explained_variance_ratio_[:5])
print(pca_df)

In [ ]:
#PCA plot
fig, ax = plt.subplots(figsize=(7, 6))

for _, row in pca_df.iterrows():
    color = "red" if row["sample"] == "G4" else ("orange" if row["group"] == "case" else "steelblue")
    ax.scatter(row["PC1"], row["PC2"], color=color, s=150)
    ax.annotate(row["sample"], (row["PC1"], row["PC2"]), fontsize=9, xytext=(5, 5), textcoords="offset points")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Sample-level PCA (pseudobulk)")
plt.tight_layout()
plt.show()